# Gaussian splat training (this repository)

Colab is **only the GPU**. Code stays in [ai-indoor-3d-reconstruction](https://github.com/prakrity26/ai-indoor-3d-reconstruction). Do not create another repo.

1. Runtime → Change runtime type → **GPU**
2. Mount Drive, then upload **IMG_7916** (or a zip of prepared `splat/images`)
3. Run all cells. The PLY is copied to Drive **in the same cell as training** so a disconnect does not wipe it.
If this notebook is newer than GitHub `main`, upload a zip of **this** repo instead of cloning, unzip it, and `os.chdir` into it. Still one repository.


In [ ]:
import os, sys, subprocess
REPO = "https://github.com/prakrity26/ai-indoor-3d-reconstruction.git"
ROOT = "/content/ai-indoor-3d-reconstruction"
if not os.path.isdir(ROOT):
    subprocess.check_call(["git", "clone", "--depth", "1", REPO, ROOT])
os.chdir(ROOT)
print("Using", ROOT)


In [ ]:
from google.colab import drive
import os, shutil

drive.mount("/content/drive")
DRIVE_SPLAT = "/content/drive/MyDrive/indoor_splat"
os.makedirs(DRIVE_SPLAT, exist_ok=True)
print("PLY will be saved to", DRIVE_SPLAT)


In [ ]:
!apt-get -qq update && apt-get -qq install -y colmap ffmpeg
!pip -q install -e ".[splat]"
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime before training"
print("cuda", torch.cuda.get_device_name(0))


In [ ]:
from google.colab import files
uploaded = files.upload()
source = next(iter(uploaded))
print("uploaded", source)


In [ ]:
import os, shutil, subprocess, sys, zipfile
job = "img7916"
if source.lower().endswith(".zip"):
    dest = f"data/frames/{job}/splat"
    os.makedirs(dest, exist_ok=True)
    zipfile.ZipFile(source).extractall(dest)
    cmd = [sys.executable, "-m", "model.splat", f"data/frames/{job}", "--job-id", job]
else:
    cmd = [sys.executable, "-m", "model.splat", source, "--job-id", job]
print(" ".join(cmd))
subprocess.check_call(cmd)
ply = f"data/frames/{job}/splat/point_cloud.ply"
assert os.path.isfile(ply), ply
kept = os.path.join(DRIVE_SPLAT, "point_cloud.ply")
shutil.copy2(ply, kept)
print("saved", kept, os.path.getsize(kept), "bytes")


In [ ]:
from google.colab import files
keep = os.path.join(DRIVE_SPLAT, "point_cloud.ply")
assert os.path.isfile(keep), keep
files.download(keep)
